In [0]:
import sys
from datetime import datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

sys.path.insert(0, "/Workspace/Users/jeremi.santoso@metrodata.co.id/test_btn/")

from config.config_load import load_config
from utils.logger       import get_logger, log_job
from utils.bronze_write import write_bronze
from utils.cdc_flag     import row_hash, cdc_flag

In [0]:
## JOB CONFIGURATION
dbutils.widgets.text("table_name", "")
TABLE_NAME = dbutils.widgets.get("table_name")
JOB_NAME   = f"bronze_{TABLE_NAME}"

BASE_PATH = "/Workspace/Users/jeremi.santoso@metrodata.co.id/test_btn/"
TABLE_CONFIG_PATH = f"{BASE_PATH}config/table_{TABLE_NAME}.yaml"

logger = get_logger(JOB_NAME)

## LOAD CONFIG
cfg   = load_config(TABLE_CONFIG_PATH)
table = cfg["tables"][TABLE_NAME]

catalog       = cfg["catalog"]
bronze_schema = cfg["schemas"]["bronze"]

bronze_cfg      = table["bronze"]
source_table    = bronze_cfg["source_table"]
target_table    = f"{catalog}.{bronze_schema}.{table['bronze_table']}"
primary_keys    = bronze_cfg["primary_keys"]
partition_col   = bronze_cfg["partition_column"]
num_partitions  = bronze_cfg.get("num_partitions", 8)
partition_by_col = bronze_cfg.get("partition_by_col", "ingested_date")

log_table = f"{catalog}.{bronze_schema}.log_jobs"

logger.info(f"Config loaded for table: {TABLE_NAME}")

In [0]:
## KONEKSI ORACLE via JDBC
oracle_host     = ""
oracle_port     = ""
oracle_service  = ""  # service name Oracle kamu
oracle_user     = dbutils.secrets.get(scope="oracle_scope", key="")
oracle_password = dbutils.secrets.get(scope="oracle_scope", key="")

oracle_jdbc_url = f"jdbc:oracle:thin:@//{oracle_host}:{oracle_port}/{oracle_service}"

In [0]:
## READ FROM ORACLE SOURCE
def read_oracle_table(spark, jdbc_url, user, password, source_table, partition_col, num_partitions=8):

    bounds_query = f"(SELECT MIN({partition_col}) as min_val, MAX({partition_col}) as max_val FROM {source_table}) t"

    bounds_df = spark.read.format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", bounds_query) \
        .option("user", user) \
        .option("password", password) \
        .option("driver", "oracle.jdbc.driver.OracleDriver") \
        .load()

    bounds = bounds_df.collect()[0]
    lower, upper = bounds["MIN_VAL"], bounds["MAX_VAL"]

    if lower is None or upper is None:
        raise ValueError(
            f"Tidak bisa menentukan lowerBound/upperBound untuk partition_col='{partition_col}' "
            f"pada table '{source_table}'. Pastikan kolom tersebut numeric dan tidak seluruhnya NULL."
        )

    df = spark.read.format("jdbc") \
        .option("url", jdbc_url) \
        .option("dbtable", source_table) \
        .option("user", user) \
        .option("password", password) \
        .option("driver", "oracle.jdbc.driver.OracleDriver") \
        .option("partitionColumn", partition_col) \
        .option("lowerBound", str(lower)) \
        .option("upperBound", str(upper)) \
        .option("numPartitions", str(num_partitions)) \
        .option("fetchsize", "10000") \
        .load()

    return df


In [0]:
def main():
    RUN_ID     = datetime.now(timezone.utc).strftime("%Y%m%d%H%M%S")
    START_TIME = datetime.now(timezone.utc)
    logger.info(f"JOB STARTED | RUN_ID={RUN_ID} | TABLE={TABLE_NAME}")
 
    try:
        logger.info(f"Reading from Oracle: {source_table}")
        df = read_oracle_table(
            spark, oracle_jdbc_url, oracle_user, oracle_password,
            source_table, partition_col, num_partitions
        )
        raw_count = df.count()
        logger.info(f"Extracted {raw_count} rows from Oracle")
 
        df = df.withColumn("ingested_timestamp", F.current_timestamp())
        df = df.withColumn("ingested_date", F.current_date())
 
        df = row_hash(
            df,
            exclude_cols=primary_keys + ["ingested_timestamp", "ingested_date"]
        )
 
        df_flagged = cdc_flag(spark, df, target_table, primary_keys)
        row_count  = df_flagged.count()
 
        counts = {r["cdc_flag"]: r["count"] for r in df_flagged.groupBy("cdc_flag").count().collect()}
        logger.info(f"CDC summary: {counts}")
 
        write_bronze(spark, df_flagged, target_table, partition_by_col, primary_keys)
 
        END_TIME = datetime.now(timezone.utc)
        elapsed  = round((END_TIME - START_TIME).total_seconds(), 2)
 
        log_job(
            spark=spark, dbutils=dbutils, table_name=log_table,
            run_id=RUN_ID, job_name=JOB_NAME, status="SUCCESS",
            start_time=START_TIME, end_time=END_TIME,
            duration=elapsed, row_count=row_count,
        )
        logger.info(f"JOB FINISHED | RUN_ID={RUN_ID} | elapsed={elapsed}s")
 
    except Exception as e:
        END_TIME = datetime.now(timezone.utc)
        elapsed  = round((END_TIME - START_TIME).total_seconds(), 2)
        logger.error(f"JOB FAILED | RUN_ID={RUN_ID} | elapsed={elapsed}s")
        logger.error(str(e))
 
        log_job(
            spark=spark, dbutils=dbutils, table_name=log_table,
            run_id=RUN_ID, job_name=JOB_NAME, status="FAILED",
            start_time=START_TIME, end_time=END_TIME,
            duration=elapsed, row_count=None, message=str(e),
        )
        raise
 
 
main()
